# Configuración en Google Colab + Google Drive

**TFM — Detección de Deepfakes**

Este notebook prepara el entorno en Colab para que **datos, figuras y modelos se
guarden en Google Drive** (persistentes entre sesiones), mientras el código se
clona desde tu repositorio.

Ejecuta las celdas **en orden**. Solo tienes que editar la URL de tu repositorio
en el paso 3.

> Antes de empezar: en el menú de Colab, **Entorno de ejecución → Cambiar tipo de
> entorno de ejecución → T4 GPU**, para entrenar con GPU.


## 1. Comprobar la GPU

In [ ]:
import torch
print("¿GPU disponible?:", torch.cuda.is_available())
print("Dispositivo:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 2. Montar Google Drive

Se abrirá una ventana para autorizar el acceso a tu Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Definir el workspace en Drive y clonar el repositorio

- El **workspace** es la carpeta de Drive donde se guardarán datos/figuras/modelos.
- El **código** se clona en el disco de Colab (efímero, se vuelve a clonar cada sesión).

✏️ **Edita `REPO_URL`** con la URL de tu repositorio.

In [ ]:
import os
from pathlib import Path

# --- Workspace persistente en Drive ---
WORKSPACE = Path('/content/drive/MyDrive/TFM_Deepfake')   # cámbialo si quieres otra ruta
for sub in ['data/raw', 'data/interim', 'data/processed', 'reports/figures', 'models']:
    (WORKSPACE / sub).mkdir(parents=True, exist_ok=True)
os.environ['TFM_WORKSPACE'] = str(WORKSPACE)   # lo leerá get_paths()

# --- Código: clonar el repositorio ---
REPO_URL = 'https://github.com/TU_USUARIO/TU_REPO.git'    # <-- EDITA ESTO
PROJECT_ROOT = Path('/content/TFM_Deepfake_Detection')
os.environ['TFM_PROJECT_ROOT'] = str(PROJECT_ROOT)

if not PROJECT_ROOT.exists():
    !git clone {REPO_URL} {PROJECT_ROOT}
else:
    !cd {PROJECT_ROOT} && git pull

print('Workspace (Drive):', WORKSPACE)
print('Proyecto (código):', PROJECT_ROOT)

## 4. Instalar dependencias

Colab ya trae `torch`, `torchvision`, `opencv`, `numpy` y `pandas` con soporte CUDA.
Solo instalamos lo que falta. `facenet-pytorch` se instala con `--no-deps` para **no
alterar la versión de PyTorch con GPU** que trae Colab.

In [ ]:
%cd {PROJECT_ROOT}
!pip install -q timm grad-cam gradio pyyaml tqdm
# facenet-pytorch sin dependencias (torch/torchvision/numpy/pillow ya están en Colab):
!pip install -q --no-deps facenet-pytorch
print('Dependencias instaladas.')

## 5. Comprobación: rutas y semillas

Verificamos que el código importa y que las rutas apuntan a Drive.

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.seeds import set_seed, load_config, get_device
from src.utils.paths import get_paths, ensure_dirs

cfg = load_config(PROJECT_ROOT / 'config' / 'config.yaml')
set_seed(cfg['project']['seed'])
paths = get_paths(cfg, PROJECT_ROOT)
ensure_dirs(paths)

print('Dispositivo:', get_device())
print('--- Rutas resueltas (deben apuntar a Drive) ---')
for k, v in paths.items():
    print(f'{k:13}: {v}')

## 6. Descargar FaceForensics++ a Drive

Necesitas el script oficial `download-FaceForensics.py` (te lo envían por email al
aceptar tu solicitud — ver `docs/01_descarga_datos.md`). Súbelo a la raíz del
proyecto en Colab (panel de archivos) o colócalo en tu Drive y cópialo.

La descarga va **directa a Drive** (`paths['raw']`). Empieza con pocos vídeos
(`-n 20`) para validar; luego amplía.

In [ ]:
RAW = paths['raw']
SCRIPT = PROJECT_ROOT / 'download-FaceForensics.py'   # ajusta si lo pones en otra ruta
import subprocess, sys

if SCRIPT.exists():
    # Solo las categorías que usa el proyecto (evita actors, DeepFakeDetection, FaceShifter):
    datasets = ['original'] + cfg['dataset']['manipulation_methods']
    for ds in datasets:
        print('Descargando:', ds)
        cmd = [sys.executable, str(SCRIPT), str(RAW), '-d', ds,
               '-c', 'c23', '-t', 'videos', '-n', '20', '--server', 'EU2']
        subprocess.run(cmd, input='\n', text=True)   # input acepta el TOS sin bloquear
else:
    print(f'No encuentro {SCRIPT}.')
    print('Sube download-FaceForensics.py (recibido por email) a la raíz del proyecto.')
    print('Recuerda colocar también los splits oficiales en:', RAW / 'splits')

## 7. Validar la extracción facial

Procesa solo los primeros vídeos para comprobar que el pipeline funciona y mirar la tasa de detección. Los recortes se guardan en Drive (`data/interim`).

In [ ]:
%cd {PROJECT_ROOT}
!python -m src.data.face_extraction --limit 20

## 8. Siguiente paso

Si todo lo anterior funciona, abre **`notebooks/01_eda.ipynb`** (detecta Colab
automáticamente y usa estas mismas rutas de Drive) y ejecútalo de principio a fin.

A partir de aquí, cada sesión de Colab solo necesita repetir los pasos 2→5
(montar Drive, clonar, instalar, comprobar); los datos ya estarán en Drive.
